In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"
SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"
CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.ingestion_file_log"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_repo_daily_activity"

In [0]:
control_df = spark.table(CONTROL_TABLE)

control_df.select(
    F.count("*").alias("logged_files"),
    F.sum(
        (F.col("file_status") == "completed").cast("int")
    ).alias("completed_files"),
    F.min("source_hour").alias("first_hour"),
    F.max("source_hour").alias("last_hour")
).show(truncate=False)

In [0]:
print("Bronze rows:", spark.table(BRONZE_TABLE).count())
print("Silver Core rows:", spark.table(SILVER_CORE_TABLE).count())
print("Gold rows:", spark.table(GOLD_TABLE).count())

In [0]:
bronze_duplicate_ids = (
    spark.table(BRONZE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

silver_duplicate_ids = (
    spark.table(SILVER_CORE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

gold_duplicate_keys = (
    spark.table(GOLD_TABLE)
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Bronze duplicate IDs:", bronze_duplicate_ids)
print("Silver duplicate IDs:", silver_duplicate_ids)
print("Gold duplicate repo/date keys:", gold_duplicate_keys)

In [0]:
hourly_volume_df = (
    spark.table(CONTROL_TABLE)
    .select(
        "source_hour",
        "row_count",
        "file_status"
    )
    .orderBy("source_hour")
)

hourly_volume_df.show(24, truncate=False)

In [0]:
import os

raw_file_sizes = []

for hour in range(24):
    file_name = f"2026-09-01-{hour}.json.gz"
    file_path = f"/Volumes/workspace/default/gharchive_raw/{file_name}"

    raw_file_sizes.append(
        (
            hour,
            file_name,
            os.path.getsize(file_path)
        )
    )

raw_file_sizes_df = spark.createDataFrame(
    raw_file_sizes,
    ["hour", "file_name", "size_bytes"]
)

raw_file_sizes_df.orderBy("hour").show(
    24,
    truncate=False
)

In [0]:
raw_file_sizes_df.select(
    F.count("*").alias("raw_files"),
    F.sum("size_bytes").alias("raw_size_bytes"),
    F.round(
        F.sum("size_bytes") / (1024 * 1024),
        2
    ).alias("raw_size_mb")
).show()

In [0]:
for table_name in [
    BRONZE_TABLE,
    SILVER_CORE_TABLE,
    GOLD_TABLE
]:
    print(table_name)

    spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select(
        "numFiles",
        "sizeInBytes"
    ).show(truncate=False)

In [0]:
daily_ingestion_df = (
    spark.table(CONTROL_TABLE)
    .withColumn(
        "source_date",
        F.to_date("source_hour")
    )
    .groupBy("source_date")
    .agg(
        F.count("*").alias("hourly_files"),
        F.sum("row_count").alias("source_rows")
    )
    .orderBy("source_date")
)

daily_ingestion_df.show(truncate=False)

In [0]:
print(
    "Completed files:",
    spark.table(CONTROL_TABLE)
    .filter(F.col("file_status") == "completed")
    .count()
)

print(
    "Bronze duplicate IDs:",
    spark.table(BRONZE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Silver duplicate IDs:",
    spark.table(SILVER_CORE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Gold duplicate repo/date keys:",
    spark.table(GOLD_TABLE)
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
print(
    "Completed files:",
    spark.table(CONTROL_TABLE)
    .filter(F.col("file_status") == "completed")
    .count()
)

print(
    "Bronze duplicate IDs:",
    spark.table(BRONZE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Silver duplicate IDs:",
    spark.table(SILVER_CORE_TABLE)
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Gold duplicate repo/date keys:",
    spark.table(GOLD_TABLE)
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
import os
from datetime import date, timedelta

raw_size_bytes = 0
raw_file_count = 0

current_date = date(2026, 9, 1)

while current_date <= date(2026, 9, 7):
    date_str = current_date.isoformat()

    for hour in range(24):
        file_name = f"{date_str}-{hour}.json.gz"
        file_path = f"/Volumes/workspace/default/gharchive_raw/{file_name}"

        if os.path.exists(file_path):
            raw_file_count += 1
            raw_size_bytes += os.path.getsize(file_path)

    current_date += timedelta(days=1)

print("Raw files:", raw_file_count)
print("Raw size bytes:", raw_size_bytes)
print("Raw size MB:", round(raw_size_bytes / (1024 * 1024), 2))
print("Raw size GB:", round(raw_size_bytes / (1024 * 1024 * 1024), 3))

In [0]:
for table_name in [
    BRONZE_TABLE,
    SILVER_CORE_TABLE,
    GOLD_TABLE
]:
    print(table_name)

    spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select(
        "numFiles",
        "sizeInBytes"
    ).show(truncate=False)

In [0]:
import time

silver_core_df = spark.table(SILVER_CORE_TABLE)

seven_day_repo_daily_df = (
    silver_core_df
    .filter(F.col("repo_id").isNotNull())
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct("actor_id").alias("active_contributors")
    )
)

start_time = time.perf_counter()

output_rows = seven_day_repo_daily_df.count()

elapsed_seconds = time.perf_counter() - start_time

print("Output rows:", output_rows)
print(
    "Execution time (seconds):",
    round(elapsed_seconds, 3)
)

## Phase 13 conclusions

The pipeline was scaled from a partial-day development dataset to seven complete days of GH Archive data.

Final scale:

- 168 hourly source files
- 5.964 GB compressed raw data
- 8,838,503 unique Bronze events
- 8,838,503 Silver Core events
- 1,785,239 repository-day Gold rows
- 13.61 GB Bronze Delta table
- 237.98 MB Silver Core Delta table
- 42.49 MB Gold Delta table
- 0 duplicate event IDs in Bronze
- 0 duplicate event IDs in Silver Core
- 0 duplicate repository-day keys in Gold

The seven-day end-to-end Databricks Job completed successfully in 41m 59s.

A representative repository-day aggregation over 8.84 million Silver rows completed in 3.175 seconds.

Seven days was selected as the final project scale because it is large enough to demonstrate incremental processing, Spark/Delta behavior, orchestration, data quality, and analytical modeling without increasing compute usage purely for a larger headline row count.